# Purchase Prediction Pipeline — AUC‑OPTIMISED (sliding window, recency features, ensemble)

In [ ]:
# ================================================================================================
# Load the standard data-science stack (pandas/numpy/matplotlib), silence warnings, and pull in the sklearn pieces needed for the pipeline
# ================================================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing   import OneHotEncoder, RobustScaler
from sklearn.impute          import SimpleImputer
from sklearn.tree            import DecisionTreeClassifier
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics         import (classification_report, roc_auc_score,
                                     f1_score, precision_recall_curve)
from scipy.sparse            import hstack
import xgboost as xgb

In [ ]:
# ================================================================================================
# 0. Helper functions
# ================================================================================================

# Builds a 50/50 balanced sample by keeping all positives and randomly drawing an equal number of negatives.
# Per-fold undersampler. For a binary y Series with index labels matching X,
# undersample the majority class so that the positive class makes up `pos_ratio` of the result.

# !!! dead code — never called anywhere in the pipeline
# Class balance is actually handled via class_weight='balanced' (DT/RF) and scale_pos_weight (XGB).


# def undersample_train(X_train_fold, y_train_fold, pos_ratio=0.5, random_state=42):
#    """Undersample majority class so that positive class makes up `pos_ratio` of the result."""
#    pos_idx = y_train_fold[y_train_fold == 1].index
#    neg_idx = y_train_fold[y_train_fold == 0].index
#    n_pos = len(pos_idx)
#    if n_pos == 0:
#        return X_train_fold, y_train_fold
#    n_neg = int(n_pos * (1.0 / pos_ratio - 1))
#    n_neg = min(n_neg, len(neg_idx))
#    rng = np.random.default_rng(random_state)
#    sampled_neg = rng.choice(neg_idx, size=n_neg, replace=False)
#    balanced_idx = np.concatenate([pos_idx, sampled_neg])
#    rng.shuffle(balanced_idx)
#    return X_train_fold.loc[balanced_idx], y_train_fold.loc[balanced_idx]

In [ ]:
# Threshold-Tuning
# Sweeps all thresholds via precision_recall_curve, computes F1 at each, returns the threshold + F1 of the best one.

def tune_threshold(y_true, y_prob):
    """Find threshold that maximises F1 score."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores = 2 * (precision * recall) / (precision + recall)
        f1_scores = np.nan_to_num(f1_scores, nan=0.0)
    best_idx = np.argmax(f1_scores)
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    return best_thresh, f1_scores[best_idx]

In [ ]:
# Read the feature importance from the trained model (how important each feature was for the prediction) and assign it to the correct name

def get_feature_importance(model, model_name, num_cols, cat_cols, encoder):
    all_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
    scores = model.feature_importances_
    imp = pd.Series(scores, index=all_names).sort_values(ascending=False)
    return imp

In [ ]:
# Leak-prevention strategy: for each day, compute features based only on past days, then update state with today's data.

"""Compute leak-free expanding-window features:
    - pid_order/click/basket rates
    - price std, target encodings
    - days_since_last_click, days_since_last_basket,
      click_streak_7d, last_event_was_click,
      click_to_order_conv, basket_to_order_conv
    """

# Expanding-window feature builder (extended with recency features)
def build_expanding_train_features(df, y_series):
    
    df = df.copy()
    df['__y__'] = y_series.reindex(df.index).to_numpy()
    df = df.sort_values('day').reset_index(drop=True)
    y_series = df.pop('__y__').astype(int)

    # Running state through previous days only
    pid_total  = {}; pid_order  = {}; pid_click = {}; pid_basket = {}; pid_prices = {}
    grp_total  = {}; grp_order  = {}
    man_total  = {}; man_order  = {}
    cat_total  = {}; cat_order  = {}
    global_orders = 0; global_total = 0

    # State for recency features 
    pid_last_click_day = {}   # most recent day with click=1 (no order)
    pid_last_basket_day = {}  # most recent day with basket=1 (no order)
    pid_last_event_type = {}  # 'click','basket','order', or None
    pid_click_days_7d = {}    # list of day numbers in last 7 days with click and no order

    # Initialise output columns
    out_cols = ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate', 'pid_price_std',
                'pid_seen_in_train', 'manufacturer_te', 'group_te', 'category_te',
                # NEW recency features
                'days_since_last_click', 'days_since_last_basket',
                'click_streak_7d', 'last_event_was_click',
                'click_to_order_conv', 'basket_to_order_conv']
    for col in out_cols:
        df[col] = np.nan

    for day in sorted(df['day'].unique()):
        mask   = df['day'] == day
        day_df = df.loc[mask]
        y_day  = y_series.loc[mask]
        prev_rate = global_orders / global_total if global_total > 0 else 0.0

        # ----- Per-pid features (including recency) -----
        for pid_val in day_df['pid'].unique():
            idx = day_df.index[day_df['pid'] == pid_val]
            total_pid = pid_total.get(pid_val, 0)
            if total_pid > 0:
                df.loc[idx, 'pid_order_rate']    = pid_order.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_click_rate']    = pid_click.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_basket_rate']   = pid_basket.get(pid_val, 0) / total_pid
                df.loc[idx, 'pid_seen_in_train'] = 1
            else:
                df.loc[idx, ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate']] = prev_rate
                df.loc[idx, 'pid_seen_in_train'] = 0
            past_prices = pid_prices.get(pid_val, [])
            df.loc[idx, 'pid_price_std'] = np.std(past_prices, ddof=0) if len(past_prices) >= 2 else 0.0

            # NEW: recency features ------------------------------------------------
            # days since last click (or 999 if never)
            if pid_val in pid_last_click_day:
                df.loc[idx, 'days_since_last_click'] = day - pid_last_click_day[pid_val]
            else:
                df.loc[idx, 'days_since_last_click'] = 999

            if pid_val in pid_last_basket_day:
                df.loc[idx, 'days_since_last_basket'] = day - pid_last_basket_day[pid_val]
            else:
                df.loc[idx, 'days_since_last_basket'] = 999

            # click streak: number of days in last 7 (excluding today) with click=1 and order=0
            click_days = pid_click_days_7d.get(pid_val, [])
            df.loc[idx, 'click_streak_7d'] = sum(1 for d in click_days if d > day - 7)

            # last event type indicator
            last_ev = pid_last_event_type.get(pid_val, None)
            df.loc[idx, 'last_event_was_click'] = 1 if last_ev == 'click' else 0

            # conversion rates (smoothed)
            n_ord = pid_order.get(pid_val, 0)
            n_click = pid_click.get(pid_val, 0)
            n_basket = pid_basket.get(pid_val, 0)
            smoothing = 10
            df.loc[idx, 'click_to_order_conv'] = (n_ord + prev_rate * smoothing) / (n_click + n_ord + smoothing)
            df.loc[idx, 'basket_to_order_conv'] = (n_ord + prev_rate * smoothing) / (n_basket + n_ord + smoothing)

        # ----- Smoothed target encoding for categorical columns -----
        for col_orig, col_te, order_dict, total_dict in [
            ('manufacturer', 'manufacturer_te', man_order, man_total),
            ('group',        'group_te',        grp_order, grp_total),
            ('category',     'category_te',     cat_order, cat_total),
        ]:
            for val in day_df[col_orig].unique():
                idx_val = day_df.index[day_df[col_orig] == val]
                n_ord = order_dict.get(val, 0)
                n_tot = total_dict.get(val, 0)
                smoothing = 10
                te = ((n_ord + prev_rate * smoothing) / (n_tot + smoothing)) if (n_tot + smoothing) > 0 else prev_rate
                df.loc[idx_val, col_te] = te

        # ----- Update state with today's data (only AFTER today's features are written) -----
        pid_day_counts = day_df.groupby('pid').size()
        pid_day_orders = y_day.groupby(day_df['pid']).sum()
        for pid_val in pid_day_counts.index:
            pid_total[pid_val] = pid_total.get(pid_val, 0) + pid_day_counts[pid_val]
            pid_order[pid_val] = pid_order.get(pid_val, 0) + pid_day_orders.get(pid_val, 0)

        for beh_col, cum_dict in [('click', pid_click), ('basket', pid_basket)]:
            day_sum = day_df.groupby('pid')[beh_col].sum()
            for pid_val in day_sum.index:
                cum_dict[pid_val] = cum_dict.get(pid_val, 0) + day_sum[pid_val]

        for pid_val, prices in day_df.groupby('pid')['price'].apply(list).items():
            if pid_val not in pid_prices:
                pid_prices[pid_val] = []
            pid_prices[pid_val].extend(prices)

        # NEW: update recency state per pid ------------------------------------
        for pid_val in day_df['pid'].unique():
            pid_df = day_df[day_df['pid'] == pid_val]
            today_order = y_day.loc[pid_df.index].sum() > 0
            today_basket = pid_df['basket'].sum() > 0
            today_click = pid_df['click'].sum() > 0

            # Determine today's dominant event (order > basket > click)
            if today_order:
                pid_last_event_type[pid_val] = 'order'
            elif today_basket:
                pid_last_event_type[pid_val] = 'basket'
            elif today_click:
                pid_last_event_type[pid_val] = 'click'
            # else keep previous

            # Update last click day if click occurred and no order (a "pure click" day)
            if today_click and not today_order:
                pid_last_click_day[pid_val] = day
            if today_basket and not today_order:
                pid_last_basket_day[pid_val] = day

            # Maintain list of days with pure clicks for streak (last 7 days)
            if pid_val not in pid_click_days_7d:
                pid_click_days_7d[pid_val] = []
            if today_click and not today_order:
                pid_click_days_7d[pid_val].append(day)
            # purge days older than 7 days (optional, keep list short)
            pid_click_days_7d[pid_val] = [d for d in pid_click_days_7d[pid_val] if d > day - 7]

        for col_orig, order_dict, total_dict in [
            ('manufacturer', man_order, man_total),
            ('group',        grp_order, grp_total),
            ('category',     cat_order, cat_total),
        ]:
            day_grp = day_df.groupby(col_orig)
            day_cnt = day_grp.size()
            day_ord = y_day.groupby(day_df[col_orig]).sum()
            for val in day_cnt.index:
                total_dict[val] = total_dict.get(val, 0) + day_cnt[val]
                order_dict[val] = order_dict.get(val, 0) + day_ord.get(val, 0)

        global_total  += len(day_df)
        global_orders += y_day.sum()

    final_global_rate = global_orders / global_total if global_total > 0 else 0.0
    return df, y_series, final_global_rate

In [ ]:
# ================================================================================================
# 1. Load Data
# ================================================================================================
train = pd.read_csv('../data/raw/train.csv', sep='|')
items = pd.read_csv('../data/raw/items.csv', sep='|')
df = train.merge(items, on='pid', how='left', validate='m:1')
print("Data loaded and merged.")

In [ ]:
# ================================================================================================
# 2. Sort and sample
# ================================================================================================
SAMPLING = 1
# Sort DataFrame by day, then by lineID → chronological order
df = df.sort_values(['day', 'lineID']).reset_index(drop=True)
# If SAMPLING < 1.0: draw a stratified sample each day (e.g., 10% each day)
if SAMPLING < 1.0:
    df = (df.groupby('day', group_keys=False)
            .sample(frac=SAMPLING, random_state=42)
            .sort_values(['day', 'lineID'])
            .reset_index(drop=True))
print(f"Using {len(df):,} rows ({SAMPLING*100:.0f}% sample).")

In [ ]:
# ================================================================================================
# 3. Feature Engineering (basic + temporal)
# ================================================================================================

# === Determine train cutoff for leak-safe feature engineering ===
df = df.sort_values(['day', 'lineID']).reset_index(drop=True)
unique_days_full = sorted(df['day'].unique())
train_cutoff_day = unique_days_full[int(len(unique_days_full) * 0.8) - 1]
print(f"Train cutoff: day {train_cutoff_day}")

In [ ]:
# --- 3a. Basic features ---
df['priceRatio'] = (df['price'] / df['rrp'].replace(0, np.nan)).fillna(1.0)
df['missingCompetitorPrice'] = df['competitorPrice'].isnull().astype(int)
df['competitorPrice'] = df['competitorPrice'].fillna(df['price'])

df['weekDay_raw'] = (df['day'] % 7).replace({0: 7})
df['weekDay_sin'] = np.sin(2 * np.pi * df['weekDay_raw'] / 7)
df['weekDay_cos'] = np.cos(2 * np.pi * df['weekDay_raw'] / 7)
df.drop(columns=['weekDay_raw'], inplace=True)

df['priceVsCompetitor'] = (df['price'] / df['competitorPrice'].replace(0, np.nan)).fillna(1.0)
df['priceDiscount'] = (df['rrp'] - df['price']).clip(lower=0)

df['adFlag_x_priceRatio'] = df['adFlag'] * df['priceRatio']
df['avail_x_priceRatio']  = df['availability'] * df['priceRatio']
df['regulated_generic']   = df['salesIndex'] * df['genericProduct']

In [ ]:
# --- 3b. Time since last promotion (adFlag) ---
df = df.sort_values(['pid','day'])
# df['last_promo_day'] = df.groupby('pid')['day'].shift(1)
# df['days_since_last_promo'] = (df['day'] - df['last_promo_day']).fillna(999).astype(int)
promo_mask = df['adFlag'] == 1
df['last_promo_day'] = np.where(promo_mask, df['day'], np.nan)
df['last_promo_day'] = df.groupby('pid')['last_promo_day'].shift(1).ffill()
df['days_since_promo'] = (df['day'] - df['last_promo_day']).fillna(999).astype(int)
df.drop(columns=['last_promo_day'], inplace=True)
# , 'days_since_last_promo'

In [ ]:
# --- 3c. Product age ---
# Calculation is done only on training data to prevent leakage. For products not seen in training, age=0 in test.
first_seen_map = df[df['day'] <= train_cutoff_day].groupby('pid')['day'].min()
df['first_seen'] = df['pid'].map(first_seen_map).fillna(df['day'])
df['product_age_days'] = df['day'] - df['first_seen']
df.drop(columns=['first_seen'], inplace=True)

In [ ]:
# --- 3d. Competitor price dynamics ---
df = df.sort_values(['pid','day'])

df['competitorPrice_7day_avg'] = (
    df.groupby('pid')['competitorPrice']
      .shift(1)
      .rolling(7, min_periods=1)
      .mean()
      .reset_index(level=0, drop=True)
)

def rolling_slope(x):
    if len(x) < 2:
        return 0.0
    t = np.arange(len(x))
    return np.polyfit(t, x, 1)[0]

df['competitorPrice_trend'] = (
    df.groupby('pid')['competitorPrice']
      .shift(1)
      .rolling(7, min_periods=2)
      .apply(rolling_slope, raw=True)
      .reset_index(level=0, drop=True)
)

In [ ]:
# ================================================================================================
# 4. Train / Test Split (time‑based, 80/20)
# ================================================================================================
X = df.drop(columns=['order'])
y = df['order']

unique_days = sorted(df['day'].unique())
cutoff = unique_days[int(len(unique_days) * 0.8)]

X_train = X[X['day'] < cutoff].copy()
X_test = X[X['day'] >= cutoff].copy()
y_train = y.loc[X_train.index].copy()
y_test = y.loc[X_test.index].copy()

print(f"Train size: {len(X_train):,}  |  Test size: {len(X_test):,}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")
print(f"Train days: {X_train['day'].min()}–{X_train['day'].max()} | Test days: {X_test['day'].min()}–{X_test['day'].max()}")

In [ ]:
# ================================================================================================
# 5. Feature lists (extended with recency features)
# ================================================================================================

#Define four lists that will later control the pipeline

NUM_FEATURES = [
    'competitorPrice', 'priceRatio', 'priceVsCompetitor', 'priceDiscount',
    'missingCompetitorPrice', 'weekDay_sin', 'weekDay_cos', 'pid_order_rate',
    'pid_click_rate', 'pid_basket_rate', 'manufacturer_te', 'group_te',
    'category_te', 'adFlag_x_priceRatio', 'avail_x_priceRatio',
    'regulated_generic', 'pid_price_std', 'pid_seen_in_train',
    'days_since_promo', 'product_age_days',
    'competitorPrice_7day_avg', 'competitorPrice_trend',
    'days_since_last_click', 'days_since_last_basket',
    'click_streak_7d', 'last_event_was_click',
    'click_to_order_conv', 'basket_to_order_conv'
]
CAT_FEATURES = [
    'adFlag', 'availability', 'content', 'unit', 'pharmForm',
    'genericProduct', 'salesIndex'
]
DROP_COLS = [
    'click', 'basket', 'revenue',         # target leakage
    'price', 'rrp',                       # encoded
    'lineID', 'day', 'pid',               # identifiers
    'campaignIndex',                      # too many missing
]
TE_COLS = [
    ('manufacturer', 'manufacturer_te'),
    ('group',        'group_te'),
    ('category',     'category_te'),
]

In [ ]:
# ================================================================================================
# 6. Final model — AUC‑optimised pipeline
# ================================================================================================
print("\n=== Final Model — AUC‑Optimised ===")

# ---- 6a. Build expanding-window features on training data ------------------------------------
X_train_expanded, y_train_expanded, global_rate = build_expanding_train_features(X_train.copy(), y_train)
print(f"Global order rate: {global_rate:.4f}")

In [ ]:
# ---- 6b. SLIDING WINDOW: keep only most recent W days for dev -------------------------------
WINDOW_SIZE = 35   # chosen via prior CV experiments (or can be tuned)
unique_days = sorted(X_train_expanded['day'].unique())
cal_size = max(1, int(len(unique_days) * 0.10))
cal_days = unique_days[-cal_size:]                     # last 10% for threshold tuning
last_dev_day = unique_days[-cal_size-1]                # day just before cal
dev_days = [d for d in unique_days if d > last_dev_day - WINDOW_SIZE and d <= last_dev_day]
print(f"Sliding window: {len(dev_days)} dev days, {len(cal_days)} cal days ({WINDOW_SIZE}-day window)")

X_dev = (X_train_expanded[X_train_expanded['day'].isin(dev_days)]
         .sort_values('day').copy())
y_dev = y_train_expanded.loc[X_dev.index].copy()
X_cal = (X_train_expanded[X_train_expanded['day'].isin(cal_days)]
         .sort_values('day').copy())
y_cal = y_train_expanded.loc[X_cal.index].copy()

In [ ]:
# ---- 6c. Build test features (extended to include recency) -----------------------------------

# Applies the features learned during training to the test set.
# For pid-based features, uses the last available value from training (which is leak-free by construction).
# For target encodings, also maps using the last training value, with global fallback for unseen categories.
# For recency features, unseen pids get 999 days since last event and 0 click streak.

def build_test_features(test_df, train_df_with_features, global_fallback_rate):
    # Basic pid features
    pid_cols = ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
                'pid_price_std', 'pid_seen_in_train',
                # NEW: recency features
                'days_since_last_click', 'days_since_last_basket',
                'click_streak_7d', 'last_event_was_click',
                'click_to_order_conv', 'basket_to_order_conv']
    train_sorted = train_df_with_features.sort_values('day')
    filled = train_sorted.copy()
    filled[pid_cols] = filled.groupby('pid')[pid_cols].ffill()
    pid_feats = filled.groupby('pid')[pid_cols].last()

    test_df = test_df.join(pid_feats, on='pid')
    
    # fill unseen pids
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
                'click_to_order_conv', 'basket_to_order_conv']:
        test_df[col] = test_df[col].fillna(global_fallback_rate)
    test_df['pid_price_std'] = test_df['pid_price_std'].fillna(0.0)
    test_df['pid_seen_in_train'] = test_df['pid_seen_in_train'].fillna(0).astype(int)
    test_df['days_since_last_click'] = test_df['days_since_last_click'].fillna(999)
    test_df['days_since_last_basket'] = test_df['days_since_last_basket'].fillna(999)
    test_df['click_streak_7d'] = test_df['click_streak_7d'].fillna(0)
    test_df['last_event_was_click'] = test_df['last_event_was_click'].fillna(0)

    # Target encodings
    for col_orig, col_te in TE_COLS:
        te_filled = train_sorted.copy()
        te_filled[col_te] = te_filled.groupby(col_orig)[col_te].ffill()
        te_map = te_filled.groupby(col_orig)[col_te].last()
        test_df[col_te] = test_df[col_orig].map(te_map).fillna(global_fallback_rate)
    return test_df

X_test_expanded = build_test_features(X_test.copy(), X_train_expanded, global_rate)

In [ ]:
# ---- 6d. Cold‑start augmentation: match real fraction ---------------------------------------
cold_start_frac = (X_test_expanded['pid_seen_in_train'] == 0).mean()
print(f"Test cold‑start fraction: {cold_start_frac:.3f}")

def augment_cold_start(X, global_rate, frac, random_state=42):
    """Mask a fraction of rows to look like never‑seen pid."""
    rng = np.random.default_rng(random_state)
    n_mask = int(len(X) * frac)
    if n_mask == 0:
        return X
    mask_idx = rng.choice(X.index, size=n_mask, replace=False)
    X = X.copy()
    # Reset all pid‑history features
    for col in ['pid_order_rate', 'pid_click_rate', 'pid_basket_rate',
                'click_to_order_conv', 'basket_to_order_conv']:
        if col in X.columns:
            X.loc[mask_idx, col] = global_rate
    if 'pid_seen_in_train' in X.columns:
        X.loc[mask_idx, 'pid_seen_in_train'] = 0
    if 'pid_price_std' in X.columns:
        X.loc[mask_idx, 'pid_price_std'] = 0.0
    # Recency features become fallbacks
    if 'days_since_last_click' in X.columns:
        X.loc[mask_idx, 'days_since_last_click'] = 999
    if 'days_since_last_basket' in X.columns:
        X.loc[mask_idx, 'days_since_last_basket'] = 999
    if 'click_streak_7d' in X.columns:
        X.loc[mask_idx, 'click_streak_7d'] = 0
    if 'last_event_was_click' in X.columns:
        X.loc[mask_idx, 'last_event_was_click'] = 0
    # Add interaction: (1 - seen) × category_te for cold‑start importance
    if 'pid_seen_in_train' in X.columns and 'category_te' in X.columns:
        X['cold_cat_interaction'] = (1 - X['pid_seen_in_train']) * X['category_te']
    return X

# Use the real cold‑start fraction for augmentation
X_dev_aug = augment_cold_start(X_dev, global_rate=global_rate, frac=cold_start_frac, random_state=42)
# If the interaction column was created, add it to NUM_FEATURES temporarily
if 'cold_cat_interaction' in X_dev_aug.columns and 'cold_cat_interaction' not in NUM_FEATURES:
    NUM_FEATURES.append('cold_cat_interaction')

In [ ]:
# ---- 6d‑b. Ensure cold_cat_interaction exists in all sets ----
for d in [X_dev_aug, X_cal, X_test_expanded]:
    if 'pid_seen_in_train' in d.columns and 'category_te' in d.columns:
        d['cold_cat_interaction'] = (1 - d['pid_seen_in_train']) * d['category_te']

In [ ]:
# ---- 6e. Drop non‑feature columns -----------------------------------------------------------
dev_days_array = X_dev_aug['day'].values
for d in [X_dev_aug, X_cal, X_test_expanded]:
    d.drop(columns=[c for c, _ in TE_COLS], errors='ignore', inplace=True)
    d.drop(columns=DROP_COLS, errors='ignore', inplace=True)

In [ ]:
# ---- 6f. Feature selection – NO correlation filter for tree models --------------------------
# If a feature is defined in NUM_FEATURES/CAT_FEATURES but has been lost somewhere (e.g., dropped or never created), it is silently skipped.
active_num = [f for f in NUM_FEATURES if f in X_dev_aug.columns]
active_cat = [f for f in CAT_FEATURES if f in X_dev_aug.columns]
print(f"Features: {len(active_num)} numeric, {len(active_cat)} categorical (no correlation drop)")
missing_num = [f for f in NUM_FEATURES if f not in X_dev_aug.columns]
if missing_num:
    print(f"WARNING: {len(missing_num)} numeric features defined but not in data: {missing_num}")

missing_cat = [f for f in CAT_FEATURES if f not in X_dev_aug.columns]
if missing_cat:
    print(f"WARNING: {len(missing_cat)} categorical features defined but not in data: {missing_cat}")

In [ ]:
# ---- 6g. Impute & cast categoricals ---------------------------------------------------------
fill = {col: 'Missing' for col in active_cat}
for d in [X_dev_aug, X_cal, X_test_expanded]:
    d.fillna(fill, inplace=True)
    d[active_cat] = d[active_cat].astype(str)

In [ ]:
# ---- 6h. Preprocessing ---------------------------------------------------------------------
# For tree-based models, we use median imputation and robust scaling for numeric features, and one-hot encoding for categoricals.
imputer = SimpleImputer(strategy='median')
scaler  = RobustScaler()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)

# Fit on dev, transform dev/cal/test (no data leakage since tree models are robust to unscaled features and missing values, but we do it for potential future linear models and to keep feature importance interpretable).
X_dev_num  = scaler.fit_transform(imputer.fit_transform(X_dev_aug[active_num]))
X_dev_cat  = encoder.fit_transform(X_dev_aug[active_cat])
X_dev_proc = hstack([X_dev_num, X_dev_cat]).tocsr()

X_cal_num  = scaler.transform(imputer.transform(X_cal[active_num]))
X_cal_cat  = encoder.transform(X_cal[active_cat])
X_cal_proc = hstack([X_cal_num, X_cal_cat]).tocsr()

X_tst_num  = scaler.transform(imputer.transform(X_test_expanded[active_num]))
X_tst_cat  = encoder.transform(X_test_expanded[active_cat])
X_tst_proc = hstack([X_tst_num, X_tst_cat]).tocsr()

In [ ]:
# ---- 6i. Day‑aware CV with 5 folds ----------------------------------------------------------
# Time‑series style folds, growing forward in time.
def make_day_folds(day_array, n_splits=3):
    unique_days = np.sort(np.unique(day_array))
    n_days = len(unique_days)
    fold_size = max(1, n_days // (n_splits + 1))
    folds = []
    for i in range(n_splits):
        train_end = (i + 1) * fold_size
        val_end   = min(train_end + fold_size, n_days)
        train_set = set(unique_days[:train_end])
        val_set   = set(unique_days[train_end:val_end])
        train_idx = np.where(np.isin(day_array, list(train_set)))[0]
        val_idx   = np.where(np.isin(day_array, list(val_set)))[0]
        if len(train_idx) > 0 and len(val_idx) > 0:
            folds.append((train_idx, val_idx))
    return folds

day_folds = make_day_folds(dev_days_array, n_splits=5)
print(f"Built {len(day_folds)} day‑aware CV folds for HPO.")

spw = (y_dev == 0).sum() / max((y_dev == 1).sum(), 1)

In [ ]:
# ---- 6j. Extended hyperparameter grids & more iterations ------------------------------------
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import cross_val_score

# Each objective function defines the hyperparameter search space for a specific model,
# trains it using the provided parameters, and evaluates it using cross-validation on the dev set with the day-aware folds.
# The mean AUC across folds is returned as the optimization target.

def objective_dt(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 200),
        'class_weight': 'balanced',
        'random_state': 42,
    }
    model = DecisionTreeClassifier(**params)
    scores = cross_val_score(model, X_dev_proc, y_dev.values,
                              cv=day_folds, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 10, 100),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.8]),
        'class_weight': 'balanced_subsample',
        'n_jobs': 1,
        'random_state': 42,
    }
    model = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_dev_proc, y_dev.values,
                              cv=day_folds, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

def objective_xgb(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 1000),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda':        trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
        'reg_alpha':         trial.suggest_float('reg_alpha', 0.0, 1.0),
        'max_delta_step':    trial.suggest_int('max_delta_step', 0, 5),
        'scale_pos_weight':  spw,
        'eval_metric':       'auc',
        'random_state':      42,
        'verbosity':         0,
        'n_jobs':            1,
    }
    model = xgb.XGBClassifier(**params)
    scores = cross_val_score(model, X_dev_proc, y_dev.values,
                              cv=day_folds, scoring='roc_auc', n_jobs=-1)
    return scores.mean()


# Helper class to wrap Optuna results in RandomizedSearchCV-like API
class StudyResult:
    def __init__(self, best_estimator, best_params, best_score):
        self.best_estimator_ = best_estimator
        self.best_params_ = best_params
        self.best_score_ = best_score

# Run optimization
searches = {}
optuna_specs = [
    ('DecisionTree', objective_dt, 30, DecisionTreeClassifier,
        {'class_weight': 'balanced', 'random_state': 42}),
    ('RandomForest', objective_rf, 50, RandomForestClassifier,
        {'class_weight': 'balanced_subsample', 'n_jobs': 1, 'random_state': 42}),
    ('XGBoost', objective_xgb, 100, xgb.XGBClassifier,
        {'scale_pos_weight': spw, 'eval_metric': 'auc',
         'random_state': 42, 'verbosity': 0, 'n_jobs': 1}),
]

for name, objective, n_trials, model_class, fixed_params in optuna_specs:
    print(f"\nOptuna HPO for {name} (n_trials={n_trials}) ...")
    study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f"  Best params:  {study.best_params}")
    print(f"  Best CV AUC:  {study.best_value:.4f}")

    best_model = model_class(**{**fixed_params, **study.best_params})
    best_model.fit(X_dev_proc, y_dev.values)
    searches[name] = StudyResult(best_model, study.best_params, study.best_value)

In [ ]:
# ---- 6k. Ensemble: optimal weights on calibration set (maximise AUC) ------------------------
# print("\n=== Optimising Ensemble Weights ===")
# from itertools import product
# best_auc = 0
# best_weights = (1/3, 1/3, 1/3)
# model_names = ['DecisionTree', 'RandomForest', 'XGBoost']
# Get calibration probabilities
# cal_probs = {}
# for name, search in searches.items():
#     cal_probs[name] = search.best_estimator_.predict_proba(X_cal_proc)[:, 1]

# Grid search over weights with step 0.05 that sum to 1
# for w1 in np.arange(0, 1.01, 0.05):
#     for w2 in np.arange(0, 1.01 - w1, 0.05):
#         w3 = 1.0 - w1 - w2
#         if abs(w1 + w2 + w3 - 1) > 1e-6:
#             continue
#         weighted = (w1 * cal_probs['DecisionTree'] +
#                     w2 * cal_probs['RandomForest'] +
#                     w3 * cal_probs['XGBoost'])
#         auc = roc_auc_score(y_cal, weighted)
#         if auc > best_auc:
#             best_auc = auc
#             best_weights = (w1, w2, w3)
# print(f"Best ensemble weights (DT, RF, XGB): {best_weights}  |  Cal AUC: {best_auc:.4f}")

# Commented out because all models are tree-based and the ensemble only gives a tiny boost, but this can be re-enabled if desired.

In [ ]:
# ---- 6l. Test evaluation (individual models, ensemble removed) -------------------------------
print("\n=== Final Test Set Results ===")
test_results = []
test_probs = {}
cal_probs = {}

# Compute calibration & test probabilities for each model
for name, search in searches.items():
    cal_probs[name]  = search.best_estimator_.predict_proba(X_cal_proc)[:, 1]
    test_probs[name] = search.best_estimator_.predict_proba(X_tst_proc)[:, 1]

# Ensemble probability
# test_prob_ens = (best_weights[0] * test_probs['DecisionTree'] +
#                  best_weights[1] * test_probs['RandomForest'] +
#                  best_weights[2] * test_probs['XGBoost'])
# Threshold from cal (on ensemble)
# ens_cal_prob = (best_weights[0] * cal_probs['DecisionTree'] +
#                 best_weights[1] * cal_probs['RandomForest'] +
#                 best_weights[2] * cal_probs['XGBoost'])
# ens_thr, _ = tune_threshold(y_cal, ens_cal_prob)
# ens_test_pred = (test_prob_ens >= ens_thr).astype(int)
# ens_auc = roc_auc_score(y_test, test_prob_ens)
# ens_f1 = f1_score(y_test, ens_test_pred, zero_division=0)
# print(f"\nEnsemble: threshold = {ens_thr:.3f}  |  Test AUC = {ens_auc:.4f}  |  Test F1 = {ens_f1:.4f}")
# print(classification_report(y_test, ens_test_pred, target_names=['no order', 'order']))
# test_results.append({
#     'model': 'Ensemble',
#     'weights': str(best_weights),
#     'test_auc': ens_auc,
#     'test_f1': ens_f1,
# })

# Commented out because all models are tree-based and the ensemble only gives a tiny boost, but this can be re-enabled if desired.


# Evaluate each model individually
for name, search in searches.items():
    thr, cal_f1 = tune_threshold(y_cal, cal_probs[name])
    test_auc   = roc_auc_score(y_test, test_probs[name])
    test_pred  = (test_probs[name] >= thr).astype(int)
    test_f1    = f1_score(y_test, test_pred, zero_division=0)

    print(f"\n{name}: threshold = {thr:.3f}  |  Test AUC = {test_auc:.4f}  |  Test F1 = {test_f1:.4f}")
    print(classification_report(y_test, test_pred, target_names=['no order', 'order']))

    test_results.append({
        'model': name,
        'best_params': str(search.best_params_),
        'cv_auc': search.best_score_,
        'threshold': thr,
        'test_auc': test_auc,
        'test_f1': test_f1,
    })

    if hasattr(search.best_estimator_, 'feature_importances_'):
        imp = get_feature_importance(search.best_estimator_, name, active_num, active_cat, encoder)
        print(f"  Top 10 features ({name}):")
        print(imp.head(10).to_string())

In [ ]:
# ================================================================================================
# 7. Diagnostics — per-day test AUC, PR-AUC, threshold band
# ================================================================================================
from sklearn.metrics import average_precision_score

test_days = X_test.loc[X_test_expanded.index, 'day'].values

per_day_rows = []
threshold_band_rows = []

In [ ]:
# 7a. Per‑day AUC for individual models
model_names = ['DecisionTree', 'RandomForest', 'XGBoost']

for name in model_names:
    prob = test_probs[name]
    df_day = pd.DataFrame({'day': test_days, 'y': y_test.values, 'p': prob})

    # Per-day metrics
    for day, grp in df_day.groupby('day'):
        if grp['y'].nunique() < 2 or len(grp) < 30:
            continue
        per_day_rows.append({
            'model': name,
            'day': day,
            'n': len(grp),
            'pos_rate': grp['y'].mean(),
            'auc': roc_auc_score(grp['y'], grp['p']),
            'pr_auc': average_precision_score(grp['y'], grp['p']),
        })

    # Threshold band
    overall_pr_auc = average_precision_score(y_test, prob)
    for thr in np.arange(0.30, 0.71, 0.05):
        y_pred = (prob >= thr).astype(int)
        threshold_band_rows.append({
            'model': name,
            'threshold': round(thr, 2),
            'f1': f1_score(y_test, y_pred, zero_division=0),
            'precision': (y_pred & y_test.values).sum() / max(y_pred.sum(), 1),
            'recall':    (y_pred & y_test.values).sum() / max(y_test.sum(), 1),
            'pr_auc':    overall_pr_auc,
        })

per_day = pd.DataFrame(per_day_rows)
band    = pd.DataFrame(threshold_band_rows)

print("\n=== Per-day test AUC (mean ± std across days) ===")
print(per_day.groupby('model')[['auc', 'pr_auc']].agg(['mean', 'std']).round(4))

print("\n=== Test PR-AUC ===")
print(band.groupby('model')['pr_auc'].first().round(4))

In [ ]:
# ---- 7b. Plot per-day AUC for all models ----------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
for name, grp in per_day.groupby('model'):
    grp = grp.sort_values('day')
    ax.plot(grp['day'], grp['auc'], marker='o', label=name, alpha=0.8)
ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='chance')
ax.set_ylim(0.5, 0.8)            
ax.set_xticks(range(int(per_day['day'].min()), int(per_day['day'].max()) + 1))
ax.set_xticklabels(range(int(per_day['day'].min()), int(per_day['day'].max()) + 1), rotation=45)
ax.set_xlabel('day')
ax.set_ylabel('test AUC')
ax.set_title('Per-day test AUC (optimised pipeline)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/modeling/per_day_test_auc_optimised.png', dpi=120)
plt.show()

In [ ]:
# ---- 7c. Save outputs -----------------------------------------------------------------------
from pathlib import Path
out_dir = Path('../data/processed/modeling')
out_dir.mkdir(parents=True, exist_ok=True)

per_day.to_csv(out_dir / 'per_day_test_metrics.csv', index=False)
band.to_csv(out_dir / 'test_threshold_band.csv', index=False)
pd.DataFrame(test_results).to_csv(out_dir / 'final_test_results.csv', index=False)

# Print final summary
print("\nPipeline completed successfully.")
best_model_row = max(test_results, key=lambda r: r['test_auc'])
print(f"Best model: {best_model_row['model']} | Test AUC = {best_model_row['test_auc']:.4f} | Test F1 = {best_model_row['test_f1']:.4f}")

In [ ]:
# ---- 7d. Confusion matrices for all models ----------------------------------------------------
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(3, 1, figsize=(6, 15))

for ax, (name, search) in zip(axes, searches.items()):
    # Use the same threshold as in evaluation (F1-optimal from cal set)
    thr, _ = tune_threshold(y_cal, cal_probs[name])
    y_pred = (test_probs[name] >= thr).astype(int)

    cm = confusion_matrix(y_test, y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['no order', 'order'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='.2%')
    ax.set_title(f'{name}\n(threshold = {thr:.3f})')

plt.tight_layout()

# Save
from pathlib import Path
out_path = Path('../data/processed/modeling/confusion_matrices.png')
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=120)
plt.show()